## Check the setup and connect to the database

In [ ]:
%run "../../scripts/01-check_setup.ipynb"

## Use HANA DataFrame with AI_Completion

A table with data already exist in your SAP HANA database, so you use [the `table()` method](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/latest/en-US/hana_ml.dataframe.html#hana_ml.dataframe.ConnectionContext.table) to instantiate a HANA DataFrame from the existing database table. 

In [ ]:
hdf_events=myconn.table('CODECONNECT', schema='VECTORS')

You can always use [the `select_statement` property](https://help.sap.com/doc/cd94b08fe2e041c2ba778374572ddba9/latest/en-US/hana_ml.dataframe.html#hana_ml.dataframe.DataFrame) to check an SQL SELECT statement that backs a HANA DataFrame. 

In [ ]:
hdf_events.select_statement

In [ ]:
hdf_events.select('*').collect()

## AI Models available to SAP HANA instance from SAP AI Core

In [ ]:
mycursor = myconn.connection.cursor()

In [ ]:
mycursor.callproc('GET_REMOTE_SOURCE_AI_MODELS', ('AI_CORE_INSTANCE', None))
models_insapaicore = mycursor.fetchall()


In [ ]:
for model in models_insapaicore:
    if 'text-generation' in model[3] and model[0]=='RUNNING': print(model[1:3])

In [ ]:
prompt='Who will present a session about integration of PAL with AI Agents'

In [ ]:
# model=""" 'gpt-4o-mini' """
model=""" '"gemini-3.5-flash"' """
# model=""" '"anthropic--claude-4.6-sonnet"' """

hdf_withprompt=hdf_events.select(
(f''' 
AI_TEXT_COMPLETION(CONCAT('Who will present a session about integration of PAL with AI Agents? ', "content_html"),
{model},
NULL,
'AI_CORE_INSTANCE') 
''', "Answer")
)

In [ ]:
print(hdf_withprompt.select_statement)

In [ ]:
import pandas as pd
pd.set_option("max_colwidth", 1024)

hdf_withprompt.collect()